# Kronos Stock Predictor — 01 数据探索

探索合成/真实 OHLCV 数据的基本特征

In [ ]:
import sys; sys.path.insert(0, '..')
import pickle, numpy as np, pandas as pd, matplotlib.pyplot as plt
from config.default_config import Config
from data.dataset import StockDataset

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100

In [ ]:
config = Config()
config.dataset_path = './data/processed'
config.batch_size = 16

ds_train = StockDataset(config.dataset_path, config, 'train')
ds_val = StockDataset(config.dataset_path, config, 'val')
ds_test = StockDataset(config.dataset_path, config, 'test')

print(f'Train: {len(ds_train)}, Val: {len(ds_val)}, Test: {len(ds_test)}')

In [ ]:
# 加载原始 test 数据查看
with open('./data/processed/test_data.pkl', 'rb') as f:
    test_data = pickle.load(f)

symbol = list(test_data.keys())[0]
df = test_data[symbol]
print(f'Symbol: {symbol}, Rows: {len(df)}')

fig, axes = plt.subplots(2, 2)
axes[0,0].plot(df.index, df['close']); axes[0,0].set_title('Close Price')
axes[0,1].plot(df.index, df['vol']); axes[0,1].set_title('Volume')
axes[1,0].plot(df.index, df['high'] - df['low']); axes[1,0].set_title('Daily Range (H-L)')
returns = df['close'].pct_change().dropna()
axes[1,1].hist(returns, bins=50, alpha=0.7); axes[1,1].set_title(f'Returns dist (std={returns.std():.4f})')
plt.tight_layout()
plt.show()

In [ ]:
# 查看 DataLoader 样本
from torch.utils.data import DataLoader
loader = DataLoader(ds_train, batch_size=4, shuffle=True)
x, stamp = next(iter(loader))
print(f'Features: {list(x.shape)}, Time: {list(stamp.shape)}')
print(f'Feature names: {config.feature_list}')
print(f'Time feature names: {config.time_feature_list}')